## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 34 samples and 9790 columns.
First few rows of the data:


,OS (days),Alive at analysis?,ID,Sex,Age,CA19-9 (U/mL),CEA (ng/mL),sample_title,sample type,cell type,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,601.0,1.0,locPDAC01,M,82.0,224.0,3.6,locPDAC01,primary human sample,purified circulating tumor cells,...,8.786438,7.488502,8.021165,7.581757,6.736391,7.178738,8.729786,8.858936,6.981255,8.318217
1,317.0,1.0,locPDAC02,F,49.0,566.0,1.1,locPDAC02,primary human sample,purified circulating tumor cells,...,6.916434,4.912205,6.026992,6.394777,4.322666,4.879872,6.906090,6.600385,5.738722,6.414802
2,208.0,1.0,locPDAC03,M,70.0,40.0,4.7,locPDAC03,primary human sample,purified circulating tumor cells,...,8.777108,6.799338,8.114119,8.431929,6.507639,6.710036,8.597269,7.912436,7.035319,8.299621
3,1139.0,1.0,locPDAC04,M,72.0,82.0,5.0,locPDAC04,primary human sample,purified circulating tumor cells,...,6.366303,4.437187,5.532977,6.174563,3.070975,4.117468,6.169198,5.700986,4.853849,6.122393
4,1016.0,1.0,locPDAC05,M,65.0,1943.0,4.8,locPDAC05,primary human sample,purified circulating tumor cells,...,5.991699,4.108643,5.419751,5.947512,3.216698,3.862012,5.950981,5.232411,4.486394,5.650229


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
